# RAG Knowledge Assistant: *Introduction to Machine Learning with Python*

This notebook builds a question-answering system that reads one book, finds the passages that matter for a question, and asks an LLM to answer using only those passages. It covers all five parts of the assignment: loading and cleaning the PDF, chunking, embeddings and a vector database, retrieval, grounded answer generation, and a small working app at the end.

**A note on the book.** The assignment mentions *Hands-On Machine Learning*, but the PDF I was given is *Introduction to Machine Learning with Python* by Müller and Guido (O'Reilly, 2017, 392 pages), so that is the one I used. It is mostly scikit-learn and classical ML, with only a short section on neural networks. That actually helps with testing: for deep-learning questions the book has little to say, so the right behaviour is "I couldn't find that in the book". I use that on purpose in Part 4.

**Stack:** `pypdf` → my own chunker → `BAAI/bge-small-en-v1.5` embeddings → FAISS → Gemini (OpenAI also supported) → Gradio.

**How to run it (Colab):** upload `intro-to-ml-book.pdf` to the Files panel, paste a Gemini API key when asked (free key from Google AI Studio), then Runtime → Run all. It takes a few minutes, mostly downloading the embedding model.

## Setup

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu google-genai openai gradio scikit-learn pandas matplotlib

In [ ]:
import os, re, json, time
import numpy as np

# ---- settings you can change ----
PDF_PATH      = "intro-to-ml-book.pdf"
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"   # 384-dim, 512-token limit, runs fine on CPU
QUERY_PREFIX  = "Represent this sentence for searching relevant passages: "  # recommended by the BGE authors for queries
CHUNK_SIZE    = 500      # tokens (counted with the embedding model's own tokenizer)
CHUNK_OVERLAP = 100      # tokens
TOP_K         = 5

LLM_PROVIDER  = os.environ.get("LLM_PROVIDER", "gemini")   # "gemini" or "openai"
LLM_MODEL     = os.environ.get("LLM_MODEL", "gemini-2.5-flash" if LLM_PROVIDER == "gemini" else "gpt-4o-mini")
INDEX_DIR     = "rag_index"

In [ ]:
# API key for the LLM (only needed from Part 4 onwards). Gemini by default; to use OpenAI set LLM_PROVIDER="openai" in the cell above.
from getpass import getpass
if LLM_PROVIDER == "gemini" and not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")
if LLM_PROVIDER == "openai" and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

# PDF: use the file if it is already in the Colab Files panel, otherwise ask for an upload
if not os.path.exists(PDF_PATH):
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = list(uploaded.keys())[0]
print("Using PDF:", PDF_PATH)

---
## Part 1: Data understanding and preprocessing

First the loading code and the cleaning rules. I wrote the cleaning rules *after* looking at the raw text (the exploration cells below show what I was reacting to), but I'm putting the functions first so the notebook reads top to bottom.

In [ ]:
from pypdf import PdfReader

def load_pdf(path):
    """Return a list of (pdf_page_number, raw_text) - page numbers start at 1."""
    reader = PdfReader(path)
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]


# Layout facts about this particular book (found while exploring it):
FIRST_BODY_PAGE = 9      # PDF page where the Preface starts (before it: cover, copyright, table of contents)
LAST_BODY_PAGE  = 380    # PDF page 381 is where the alphabetical index begins
PRINTED_OFFSET  = 14     # printed page number = PDF page - 14 (Chapter 1 starts on PDF page 15 = printed page 1)

CHAPTER_TITLES = {
    1: "Introduction",
    2: "Supervised Learning",
    3: "Unsupervised Learning and Preprocessing",
    4: "Representing Data and Engineering Features",
    5: "Model Evaluation and Improvement",
    6: "Algorithm Chains and Pipelines",
    7: "Working with Text Data",
    8: "Wrapping Up",
}

# Every page ends with a running footer, e.g. "Supervised Machine Learning Algorithms | 87",
# "88 | Chapter 2: Supervised Learning", or just a bare page number on chapter-opening pages.
FOOTER_RE = re.compile(r"^(?:(?:\d+|[ivxlc]+)\s*\|\s*.+|.+\s*\|\s*(?:\d+|[ivxlc]+)|\d+|[ivxlc]+)$")


def clean_page(text):
    """Remove the footer, re-join hyphenated words and fix the 'Y ou' kerning glitch.
    Code is left alone on purpose (indentation matters and it is useful to retrieve)."""
    lines = text.rstrip().split("\n")
    if lines and FOOTER_RE.match(lines[-1].strip()):
        lines.pop()
    text = "\n".join(lines)
    text = text.replace("\u2010\n", "")                       # "classifica-\ntion" -> "classification"
    text = re.sub(r"\bY (ou(?:r|rs|rself|rselves|Tube)?)\b", r"Y\1", text)  # "Y ou" -> "You"
    text = re.sub(r"[ \t]+\n", "\n", text)                    # trailing spaces
    return text.strip()


def page_label(start, end):
    """Printed page label used in citations."""
    if end < 1:
        return "front matter"
    start = max(start, 1)
    return str(start) if start == end else f"{start}-{end}"


def build_lines(raw_pages):
    """Turn raw PDF pages into a list of body lines, each tagged with chapter and page.
    Returns: list of dicts {text, chapter, pdf_page}"""
    lines, chapter = [], "Preface"
    dangling = None                       # a word that was split across a page break
    for pno, raw in raw_pages:
        if pno < FIRST_BODY_PAGE or pno > LAST_BODY_PAGE:
            continue
        m = re.match(r"\s*CHAPTER (\d+)\b", raw)
        if m:
            chapter = f"Chapter {m.group(1)}: {CHAPTER_TITLES[int(m.group(1))]}"
        cleaned = clean_page(raw)
        if len(cleaned) < 50:             # blank / part-divider pages
            continue
        page_lines = [ln for ln in cleaned.split("\n") if ln.strip()]
        if dangling is not None:          # glue "evalu-" + "ation ..." back together
            dangling["text"] = dangling["text"][:-1] + page_lines.pop(0)
            dangling = None
        for ln in page_lines:
            lines.append({"text": ln, "chapter": chapter, "pdf_page": pno})
        if lines and lines[-1]["text"].endswith("\u2010"):
            dangling = lines[-1]
    return lines

In [ ]:
raw_pages = load_pdf(PDF_PATH)
texts = [t for _, t in raw_pages]
lens = [len(t) for t in texts]
all_text = "\n".join(texts)

print("Pages:", len(raw_pages))
print("Characters:", f"{sum(lens):,}", "| average per page:", round(np.mean(lens)))
print("Near-empty pages (<50 chars):", [p for p, t in raw_pages if len(t.strip()) < 50])
print()
print("Words split at a line end with a special hyphen (U+2010):", all_text.count("\u2010\n"))
print("'Y ou' kerning glitches:", len(re.findall(r"\bY (?:ou|our)", all_text)))
print("Dotted table-of-contents lines:", len(re.findall(r"(?:\. ){4,}", all_text)))
print("Figure captions:", len(re.findall(r"^Figure \d+-\d+\.", all_text, re.M)))
print("Code cells (In[n]: markers):", len(re.findall(r"^In\[\d+\]:", all_text, re.M)))

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.bar(range(1, len(lens) + 1), lens, width=1.0)
plt.xlabel("PDF page"); plt.ylabel("characters"); plt.title("Extracted text per page")
plt.axvline(FIRST_BODY_PAGE, color="r", ls="--"); plt.axvline(LAST_BODY_PAGE, color="r", ls="--")
plt.tight_layout(); plt.show()

**What I found in the text**

- The PDF has a proper text layer, so no OCR was needed. `pypdf` reads all 392 pages in a few seconds.
- 5 pages come back empty. They are blank pages, mostly the left-hand page just before a chapter opener.
- **Running footer on every page**, like `Supervised Machine Learning Algorithms | 87` or `88 | Chapter 2: Supervised Learning`. If I left these in, they would end up inside chunks as noise, so I strip the last line of each page when it matches that pattern.
- **Hyphenated words:** 736 words are split across lines with a special hyphen character (`classifica‐` / `tion`). A tokenizer would see two meaningless fragments, so I glue them back together. I first handled only line-end hyphens and then noticed 7 leftovers; these were words split across a *page break*, with a footnote sitting in between. So the cleaner now also joins a dangling word to the first line of the next page.
- **Kerning glitch:** "You" is extracted as "Y ou" (107 times), because of the font spacing. Easy regex fix, but worth doing since "you" is such a common word.
- **Front matter and index:** the table of contents is full of dot leaders and the index is just terms and page numbers. Neither answers questions, and the index would match almost any keyword query, so I keep only the body (PDF pages 9 to 380, Preface through Chapter 8).
- **Code:** I decided to *keep* the code cells (`In[12]:` ...) and not touch indentation. Questions like "how do I use GridSearchCV" are often best answered by a code example in the book.
- **Figures:** the 156 captions are kept as text, but the plots themselves are images, so they can't be retrieved. That's a real limitation, and I come back to it at the end.

Here is one page before and after cleaning:

In [ ]:
pno = 101   # printed page 87 (random forests)
raw = dict(raw_pages)[pno]
print("---------- RAW (last 350 chars) ----------")
print(raw[-350:])
print("\n---------- CLEANED (last 350 chars) ----------")
print(clean_page(raw)[-350:])

### Chunking

The assignment suggests 500 to 1000 tokens with 50 to 150 overlap, "or more based on experimentation". My choices:

- **Size 500, overlap 100.** Embedding models have a maximum input length, and anything beyond it is silently cut off. `bge-small` accepts 512 tokens, so 500 is the biggest chunk that fits completely. Going bigger would only help with a longer-context embedding model.
- **Tokens are counted with the embedding model's own tokenizer**, not by words or characters. That way "500" really means 500 to the model that will read it.
- **Chunks follow the book's lines, not arbitrary character positions**, and each chunk tries to end at the end of a sentence (when one falls in the last 40% of the window) so ideas aren't chopped in half. Overlap is made of whole lines too.
- **Chunks never cross a chapter boundary**, and every chunk remembers its chapter and printed page numbers, so answers can cite where they came from.

I compare 500/100 against a smaller 250/50 setting in Part 3.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
embedder.max_seq_length = 512
tokenizer = embedder.tokenizer

def count_tokens(strings):
    return [len(ids) for ids in tokenizer(strings, add_special_tokens=False)["input_ids"]]

In [ ]:
SENT_END = re.compile(r"[.!?:)\]]\s*$")

def chunk_lines(lines, count_tokens, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Pack lines into chunks of at most `size` tokens with ~`overlap` tokens shared between
    neighbours. Chunks never cross a chapter boundary, and a chunk prefers to stop at the end
    of a sentence (if one falls in the last 40% of the window) instead of mid-sentence.

    count_tokens: function taking a list of strings and returning a list of token counts."""
    chunks = []
    # group consecutive lines by chapter
    groups, current = [], []
    for ln in lines:
        if current and ln["chapter"] != current[0]["chapter"]:
            groups.append(current); current = []
        current.append(ln)
    if current:
        groups.append(current)

    for grp in groups:
        tok = count_tokens([l["text"] for l in grp])
        n, i = len(grp), 0
        while i < n:
            j, total = i, 0
            while j < n and total + tok[j] <= size:
                total += tok[j]; j += 1
            if j == i:                       # one enormous line - take it alone
                j = i + 1
            cut = j
            if j < n:                        # try to end on a sentence boundary
                for k in range(j, i, -1):
                    if sum(tok[i:k]) < 0.6 * size:
                        break
                    if SENT_END.search(grp[k - 1]["text"]):
                        cut = k; break
            piece = grp[i:cut]
            start, end = piece[0]["pdf_page"] - PRINTED_OFFSET, piece[-1]["pdf_page"] - PRINTED_OFFSET
            chunks.append({
                "id": len(chunks),
                "chapter": grp[0]["chapter"],
                "page_start": start, "page_end": end,
                "page_label": page_label(start, end),
                "n_tokens": sum(tok[i:cut]),
                "text": "\n".join(l["text"] for l in piece),
            })
            if cut >= n:
                break
            nxt, acc = cut, 0                # step back to create the overlap
            while nxt - 1 > i and acc + tok[nxt - 1] <= overlap:
                nxt -= 1; acc += tok[nxt]
            i = nxt
    return chunks

In [ ]:
lines = build_lines(raw_pages)
chunks = chunk_lines(lines, count_tokens, CHUNK_SIZE, CHUNK_OVERLAP)

sizes = [c["n_tokens"] for c in chunks]
print(f"{len(lines):,} body lines -> {len(chunks)} chunks")
print(f"tokens per chunk: mean {np.mean(sizes):.0f}, min {min(sizes)}, max {max(sizes)}")

# sanity check: re-tokenise the final chunk text (with the special tokens the model adds) - nothing may exceed 512
full_lengths = [len(ids) for ids in tokenizer([c["text"] for c in chunks])["input_ids"]]
print("longest chunk incl. special tokens:", max(full_lengths), "(limit 512)")

plt.figure(figsize=(7, 3)); plt.hist(sizes, bins=30); plt.xlabel("tokens per chunk"); plt.ylabel("chunks"); plt.tight_layout(); plt.show()

c = chunks[150]
print(f"\nExample chunk #{c['id']} - {c['chapter']}, p. {c['page_label']}, {c['n_tokens']} tokens\n")
print(c["text"][:900], "...")

---
## Part 2: Embeddings and vector database

**Embedding model: `BAAI/bge-small-en-v1.5`** (SentenceTransformers). It turns a piece of text into a 384-number vector so that texts with similar meaning end up close together. I picked it because it runs locally on a CPU with no API key or cost, it is small (about 33M parameters) but does well on retrieval benchmarks for its size, and it accepts 512 tokens, which fits my chunk size. For *queries*, the BGE authors recommend a short instruction prefix ("Represent this sentence for searching relevant passages: "), so I add it to the question but not to the book chunks.

**Vector storage: FAISS `IndexFlatIP`.** All vectors are normalised to length 1, so the inner product between two vectors is exactly their cosine similarity. `IndexFlatIP` does an exact search, comparing the query with every vector. With only a few hundred chunks that takes well under a millisecond, so there is no reason to use an approximate index (IVF, HNSW), which only pays off at millions of vectors. The chunk text and metadata (chapter, pages) are kept in a parallel list, so row *i* of the index is chunk *i*. Both are saved to the `rag_index/` folder so I don't have to re-embed every time.

In [ ]:
class Retriever:
    """Embeds the chunks, keeps them in a FAISS index and answers similarity searches."""

    def __init__(self, embedder, chunks, embeddings=None):
        import faiss
        self.embedder, self.chunks = embedder, chunks
        if embeddings is None:
            embeddings = embedder.encode([c["text"] for c in chunks], batch_size=32,
                                         normalize_embeddings=True, show_progress_bar=True)
        embeddings = np.asarray(embeddings, dtype="float32")
        self.index = faiss.IndexFlatIP(embeddings.shape[1])   # inner product == cosine for unit vectors
        self.index.add(embeddings)
        self.embeddings = embeddings

    def search(self, query, k=TOP_K):
        q = self.embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return [dict(self.chunks[i], score=float(s), rank=r + 1)
                for r, (s, i) in enumerate(zip(scores[0], ids[0])) if i != -1]

    def save(self, folder=INDEX_DIR):
        os.makedirs(folder, exist_ok=True)
        np.save(os.path.join(folder, "embeddings.npy"), self.embeddings)
        with open(os.path.join(folder, "chunks.json"), "w") as f:
            json.dump(self.chunks, f)

    @classmethod
    def load(cls, embedder, folder=INDEX_DIR):
        with open(os.path.join(folder, "chunks.json")) as f:
            chunks = json.load(f)
        return cls(embedder, chunks, np.load(os.path.join(folder, "embeddings.npy")))

In [ ]:
retriever = Retriever(embedder, chunks)
retriever.save()
print("Vectors in index:", retriever.index.ntotal, "| dimension:", retriever.embeddings.shape[1])

In [ ]:
# Example: what does the retriever return for a sample query?
query = "Why does a random forest overfit less than a single decision tree?"
for h in retriever.search(query, k=3):
    print(f"[rank {h['rank']}] score={h['score']:.3f} | {h['chapter']} | p. {h['page_label']}")
    print(h["text"][:500].replace("\n", " "), "...\n")

---
## Part 3: Retrieval pipeline

`retriever.search(query, k)` already does the three steps the assignment lists: embed the query, search the index, return the top-k chunks with their scores.

To pick k (and to check my chunking choice) with evidence instead of gut feeling, I wrote 18 test questions covering every chapter, and noted by hand the printed pages where each answer is explained. A retrieval counts as a **hit@k** if at least one of the top-k chunks overlaps those pages. **MRR** (mean reciprocal rank) also rewards putting the right chunk near the top. It's a small test set, so I treat the numbers as a guide, not a benchmark.

In [ ]:
# Hand-made evaluation set: (question, (first_printed_page, last_printed_page) where the answer lives)
EVAL_SET = [
    ("What is the difference between supervised and unsupervised learning?", (1, 4)),
    ("What are overfitting and underfitting, and how do they relate to model complexity?", (27, 29)),
    ("How does the k-nearest neighbors algorithm make a prediction?", (35, 37)),
    ("What does the alpha parameter do in ridge regression?", (49, 52)),
    ("How is lasso different from ridge regression?", (53, 55)),
    ("Why does a random forest overfit less than a single decision tree?", (83, 87)),
    ("How do gradient boosted regression trees work and which parameters control them?", (88, 92)),
    ("What is the kernel trick in support vector machines?", (96, 98)),
    ("How does principal component analysis find the principal components?", (146, 152)),
    ("What are the weaknesses of k-means clustering?", (170, 181)),
    ("What is DBSCAN and which parameters does it have?", (187, 190)),
    ("Why is cross-validation better than a single train/test split?", (252, 257)),
    ("What is grid search and how does it avoid overfitting to the test set?", (260, 267)),
    ("What do the ROC curve and AUC measure?", (293, 297)),
    ("How do pipelines help prevent information leakage during preprocessing?", (308, 315)),
    ("How does one-hot encoding represent categorical variables?", (213, 216)),
    ("What is the bag-of-words representation of text?", (327, 329)),
    ("What is tf-idf scaling?", (336, 338)),
]

def page_hit(chunk, gold):
    lo, hi = gold
    return chunk["page_start"] <= hi and chunk["page_end"] >= lo

def evaluate(search_fn, ks=(3, 5, 7)):
    """search_fn(query, k) -> list of chunks. A question counts as a hit@k if any of the
    top-k chunks overlaps the pages where the answer is. MRR rewards ranking the hit higher."""
    kmax = max(ks)
    first_hit = []
    for q, gold in EVAL_SET:
        hits = search_fn(q, kmax)
        first_hit.append(next((r for r, h in enumerate(hits, 1) if page_hit(h, gold)), None))
    row = {f"hit@{k}": round(float(np.mean([r is not None and r <= k for r in first_hit])), 2) for k in ks}
    row["MRR"] = round(float(np.mean([1 / r if r else 0 for r in first_hit])), 3)
    return row

In [ ]:
import pandas as pd

# 1) k = 3, 5, 7 with the main setup
print("Dense retrieval (bge-small), chunk 500 / overlap 100")
print(pd.DataFrame([evaluate(retriever.search)], index=["bge-small"]).to_string())

# 2) which questions miss, and at which rank is the first correct chunk?
rows = []
for q, gold in EVAL_SET:
    hits = retriever.search(q, 7)
    first = next((r for r, h in enumerate(hits, 1) if page_hit(h, gold)), None)
    rows.append({"question": q[:70], "answer pages": f"{gold[0]}-{gold[1]}", "first correct rank": first or "not in top 7"})
pd.DataFrame(rows)

In [ ]:
# 3) chunk size experiment + a plain keyword (TF-IDF) baseline for comparison
from sklearn.feature_extraction.text import TfidfVectorizer

results = {}
results["bge-small, 500/100"] = evaluate(retriever.search)

chunks_small = chunk_lines(lines, count_tokens, 250, 50)
retriever_small = Retriever(embedder, chunks_small)
results["bge-small, 250/50"] = evaluate(retriever_small.search)

tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
X = tfidf.fit_transform([c["text"] for c in chunks])
def tfidf_search(q, k):
    scores = (X @ tfidf.transform([q]).T).toarray().ravel()
    return [chunks[i] for i in scores.argsort()[::-1][:k]]
results["TF-IDF keywords, 500/100"] = evaluate(tfidf_search)

print(f"chunks: 500/100 -> {len(chunks)}, 250/50 -> {len(chunks_small)}")
pd.DataFrame(results).T

**Reading the results** (fill in with your own numbers after running):

- *Effect of k:* a larger k can only raise hit@k, because it adds more chances to include the right passage. The cost is more text sent to the LLM, which means more noise, higher cost, and a higher chance that the model gets distracted by a loosely related passage. I look for the point where hit@k stops improving much and use that as the default (the app uses k = 5, and the slider lets you change it).
- *Effect of chunk size:* smaller chunks are more precise but often cut an explanation in two, while larger chunks keep a whole explanation together but dilute the embedding with other topics. The table above shows which won on this book.
- *Dense vs keyword baseline:* the TF-IDF row shows how much the embedding model actually adds. On a technical book full of exact terms (`GridSearchCV`, `alpha`, `n_estimators`) a keyword baseline is strong, and the gap shows up mostly on paraphrased questions. Combining both ("hybrid search") would be a natural next step.

> ✏️ **Add your own 2 to 3 lines here** with the actual numbers from your run, e.g. "hit@5 was X with 500/100 versus Y with 250/50, so I kept ...".

---
## Part 4: Answer generation (RAG)

**Prompt design.** The prompt has two layers. A *system prompt* sets the rules, and the *user message* carries the retrieved passages and the question. The reasoning behind each rule:

1. **"Use ONLY the numbered passages."** This is the main anti-hallucination rule. LLMs love to fill gaps from memory, and the prompt explicitly forbids it, even when the model knows the answer.
2. **A fixed refusal sentence** ("I couldn't find that in the book.") when the passages don't contain the answer. A fixed sentence is easy to spot, and it gives the model an acceptable way out instead of inventing something.
3. **Numbered passages with chapter and page, and a rule to cite them like `[2]`.** This makes every claim traceable, and I can check programmatically which passages were actually used.
4. **Partial answers allowed**, with a note about what's missing, so the model doesn't refuse a question it can half answer.
5. **Short answers, code only if it's in the passages.** Less room to wander, and no invented code.
6. **Temperature 0**, so the output stays as close to the context as possible, and the same question gives the same answer.
7. The question comes **last**, after the passages, so it's the freshest thing in the prompt.

**Evidence of grounding.** For every answer I show (a) the passages that were retrieved, with chapter, page and similarity score, (b) which passage numbers the answer cited, and (c) a simple *context word overlap*: the share of the answer's content words that also appear in the retrieved text. That last one is only a rough lexical proxy and cannot prove an answer is correct, so I also read the cited passages myself.

In [ ]:
SYSTEM_PROMPT = """You are a teaching assistant for the book "Introduction to Machine Learning with Python" by Andreas Mueller and Sarah Guido.

Answer the student's question using ONLY the numbered context passages that are provided.

Rules:
1. If the passages do not contain enough information to answer the question, reply exactly: "I couldn't find that in the book." and stop. Do not guess and do not use outside knowledge, even if you know the answer.
2. After each claim, cite the passage(s) it came from with the passage number in square brackets, like [1] or [2][4].
3. Keep the answer short and clear: one short paragraph, or a few bullet points if the question has several parts. Only include code if it appears in the passages.
4. If the passages answer only part of the question, answer that part and say what is missing."""


def build_prompt(query, hits):
    blocks = [f"[{n}] ({h['chapter']}, p. {h['page_label']})\n{h['text']}" for n, h in enumerate(hits, 1)]
    return ("Context passages:\n\n" + "\n\n".join(blocks) +
            f"\n\nQuestion: {query}\n\nAnswer (use only the passages above and cite them like [1]):")


def generate(prompt):
    """Send the prompt to the LLM. temperature=0 so answers stay as close to the context as possible."""
    if LLM_PROVIDER == "gemini":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        resp = client.models.generate_content(
            model=LLM_MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.0))
        return resp.text.strip()
    if LLM_PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()                      # reads OPENAI_API_KEY
        resp = client.chat.completions.create(
            model=LLM_MODEL, temperature=0,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}])
        return resp.choices[0].message.content.strip()
    raise ValueError("LLM_PROVIDER must be 'gemini' or 'openai'")


_STOP = set("this that with from have been were will would could should also which their there these those about into than then them they what when where while your does done only more most such some other being because between using used uses each both very much many".split())

def grounding_report(answer, hits):
    """Cheap grounding evidence: which passages were cited, and how many of the answer's content
    words actually occur in the retrieved text (a lexical proxy, not a proof)."""
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 0 < int(n) <= len(hits)})
    context_words = set(re.findall(r"[a-z_]{4,}", " ".join(h["text"] for h in hits).lower()))
    answer_words = [w for w in re.findall(r"[a-z_]{4,}", re.sub(r"\[\d+\]", "", answer.lower())) if w not in _STOP]
    overlap = sum(w in context_words for w in answer_words) / max(1, len(answer_words))
    return {"cited_passages": cited, "context_word_overlap": round(overlap, 2)}


def ask(query, k=TOP_K):
    hits = retriever.search(query, k)
    answer = generate(build_prompt(query, hits))
    return {"query": query, "answer": answer, "sources": hits, "grounding": grounding_report(answer, hits)}


def show(result, preview_chars=220):
    print("Q:", result["query"], "\n")
    print("A:", result["answer"], "\n")
    print("Grounding:", result["grounding"])
    print("Retrieved passages:")
    for i, h in enumerate(result["sources"], 1):
        snippet = " ".join(h["text"].split())[:preview_chars]
        print(f"  [{i}] {h['chapter']} | p. {h['page_label']} | score {h['score']:.3f}\n      {snippet}...")

In [ ]:
# In-scope questions: the book should contain the answer
questions = [
    "What is the difference between a random forest and gradient boosted trees?",
    "Why should I put the scaler inside a Pipeline when using cross-validation?",
    "How do I choose between L1 and L2 regularization for a linear model?",
    "What is the bag-of-words representation and what are its limitations?",
]
for q in questions:
    show(ask(q, k=5))
    print("\n" + "=" * 100 + "\n")

In [ ]:
# Out-of-scope questions: a grounded assistant should refuse instead of making something up
out_of_scope = [
    "Who won the FIFA World Cup in 2022?",
    "How does backpropagation work in a convolutional neural network?",
    "Explain how the attention mechanism in a transformer works.",
]
for q in out_of_scope:
    res = ask(q, k=5)
    print("Q:", q)
    print("A:", res["answer"])
    print("   top retrieval score:", round(res["sources"][0]["score"], 3), "| grounding:", res["grounding"], "\n")

**What to look for in the outputs above**

- For the in-scope questions, each claim has a bracket citation, the cited passages really do contain that material (open the passage and check), and the context-word overlap is high.
- For the out-of-scope questions, the answer should be the refusal sentence. The deep-learning questions are the interesting ones: the book touches on neural networks, so the retriever *will* find loosely related passages (look at the scores), and it's the prompt, not the retriever, that stops the model from answering from its own knowledge.
- The similarity scores of out-of-scope questions tend to be lower than for in-scope ones, so a minimum-score cutoff could be added as a second safety net before calling the LLM. I haven't enabled one because the right threshold should be picked by looking at real scores from your run.

> ✏️ **Add a sentence or two here** about what you observed, for example whether any answer cited a passage that didn't really support it.

---
## Part 5: End-to-end application

Everything is now connected: PDF → cleaning → chunking → embeddings → FAISS → prompt → LLM. The cell below launches a small Gradio web app for real-time questions. It shows the answer, the passages it came from, and the grounding info. There is also a plain `app.py` (same code) so the project can run outside Colab.

In [ ]:
def launch_app(share=False):
    """Small web UI: type a question, get a cited answer plus the passages it was based on."""
    import gradio as gr

    def respond(question, k):
        if not question or not question.strip():
            return "Please type a question first.", ""
        res = ask(question.strip(), int(k))
        sources = "\n\n".join(
            f"**[{i}] {h['chapter']}, p. {h['page_label']}** (similarity {h['score']:.2f})\n\n> "
            + " ".join(h["text"].split())[:350] + "..."
            for i, h in enumerate(res["sources"], 1))
        g = res["grounding"]
        note = f"\n\n---\n*Cited passages: {g['cited_passages']} | answer words found in context: {g['context_word_overlap']:.0%}*"
        return res["answer"] + note, sources

    demo = gr.Interface(
        fn=respond,
        inputs=[gr.Textbox(label="Your question", lines=2, placeholder="e.g. Why is cross-validation better than a single split?"),
                gr.Slider(1, 10, value=TOP_K, step=1, label="Passages to retrieve (k)")],
        outputs=[gr.Markdown(label="Answer"), gr.Markdown(label="Retrieved passages")],
        title="ML Knowledge Assistant",
        description="Answers come only from 'Introduction to Machine Learning with Python' (Mueller & Guido).",
        examples=[["What is the difference between a random forest and gradient boosted trees?", 5],
                  ["Why should I use a Pipeline together with GridSearchCV?", 5],
                  ["How do I choose the number of clusters for k-means?", 5]])
    demo.launch(share=share)

In [ ]:
launch_app(share=True)   # share=True gives a public link when running in Colab

In [ ]:
# Optional: a simple chat loop inside the notebook (press Enter on an empty line to stop)
while True:
    q = input("\nAsk the book (empty to quit) > ").strip()
    if not q:
        break
    show(ask(q))

### Demonstration of multiple queries

Use the app (or the loop above) to try questions of different types. The ones I used for testing:

| Type | Example question |
|---|---|
| Definition | What is overfitting? |
| Comparison | How is lasso different from ridge regression? |
| How-to / code | How do I tune SVM parameters with GridSearchCV? |
| Multi-chapter | How do I evaluate a classifier on imbalanced data? |
| Out of scope | Who is the current president of the USA? (should be refused) |

---
## Limitations and what I would do next

- **Figures are lost.** Plots are images, so a question like "what does the decision boundary look like" can only be answered from the caption and surrounding text. A multimodal model or OCR on figures would be needed.
- **Equations and code layout** come out of the PDF as plain text and are sometimes hard to read.
- **Retrieval is dense only.** Adding keyword search (BM25) and merging the two rankings usually helps on technical terms, and a cross-encoder re-ranker could improve the order of the top results.
- **The evaluation set is small** (18 questions that I wrote myself), and the grounding check is lexical. A better check would have the LLM verify each claim against the cited passage.
- **No conversation memory.** Each question is answered independently, so follow-ups like "and what about its weaknesses?" won't work yet.
- **Chunk size is capped by the embedding model (512 tokens).** A longer-context embedding model would allow the 800 to 1000 token chunks mentioned in the assignment, and would be worth testing.